In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, roc_auc_score, f1_score
import numpy as np
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
DATA_DIR = r"C:\CliniScan\classification_data"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

In [4]:
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [5]:
train_dataset = ImageFolder(TRAIN_DIR, transform=transform_train)
val_dataset = ImageFolder(VAL_DIR, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Classes:", class_names)

Train dataset size: 11301
Val dataset size: 2826
Classes: ['Abnormal', 'Normal']


In [6]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [8]:
EPOCHS = 10

best_val_acc = 0

In [9]:
for epoch in range(EPOCHS):

    model.train()
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc="Training"):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total


    model.eval()

    val_correct = 0
    val_total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)[:,1]

            _, predicted = torch.max(outputs,1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_acc = 100 * val_correct / val_total

    f1 = f1_score(all_labels, all_preds)

    auc = roc_auc_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Val Accuracy: {val_acc:.2f}%")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC Score: {auc:.4f}")


    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(model.state_dict(), "classifier_lr_0001.pth")

        print("Best model saved.")

    scheduler.step()

Training: 100%|██████████| 354/354 [02:26<00:00,  2.42it/s]



Epoch 1/10
Train Accuracy: 89.82%
Val Accuracy: 92.32%
F1 Score: 0.9460
AUC Score: 0.9749
Best model saved.


Training: 100%|██████████| 354/354 [02:20<00:00,  2.52it/s]



Epoch 2/10
Train Accuracy: 92.84%
Val Accuracy: 91.47%
F1 Score: 0.9382
AUC Score: 0.9721


Training: 100%|██████████| 354/354 [02:23<00:00,  2.47it/s]


...


Epoch 8/10
Train Accuracy: 97.50%
Val Accuracy: 94.55%
F1 Score: 0.9617
AUC Score: 0.9879


Training: 100%|██████████| 354/354 [02:25<00:00,  2.43it/s]



Epoch 9/10
Train Accuracy: 97.98%
Val Accuracy: 94.09%
F1 Score: 0.9583
AUC Score: 0.9865


Training: 100%|██████████| 354/354 [02:25<00:00,  2.44it/s]



Epoch 10/10
Train Accuracy: 98.00%
Val Accuracy: 94.59%
F1 Score: 0.9621
AUC Score: 0.9868


In [10]:
print("\nFinal Evaluation")

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))


Final Evaluation
              precision    recall  f1-score   support

    Abnormal       0.94      0.87      0.91       836
      Normal       0.95      0.98      0.96      1990

    accuracy                           0.95      2826
   macro avg       0.94      0.92      0.93      2826
weighted avg       0.95      0.95      0.95      2826

